In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
import os
from datetime import datetime

## Static Models

In [2]:
df = pd.read_parquet("../data/data_cleaned.parquet")

df_merge = df.copy()
df_merge['approval_year'] = df['approvaldate'].dt.year
df_merge['approval_month'] = df_merge['approvaldate'].dt.month

MACRO_DIR = '../data/macro_clean'

In [3]:
# GDP: shift 1 year, merge by state; groupby(state) ffill
fname = 'gdp_yearly_by_state.csv'
gdp_path = os.path.join(MACRO_DIR, fname)
gdp = pd.read_csv(gdp_path)
gdp = gdp.sort_values(['state', 'year'])
gdp['GDP_y_st'] = gdp.groupby('state')['GDP'].ffill()
gdp['GDP_growth_y_st'] = gdp.groupby('state')['GDP_y_st'].pct_change()
gdp = gdp.drop(columns=['GDP'])
df_merge['gdp_lookup_year'] = df_merge['approval_year'] - 1
df_merge = df_merge.merge(
    gdp,
    left_on=['gdp_lookup_year', 'borrstate'],
    right_on=['year', 'state'],
    how='left',
    suffixes=('', '_gdp'),
    validate="m:1"
)
df_merge = df_merge.drop(columns=['gdp_lookup_year', 'year', 'state'], errors='ignore')
print(f'Merged {fname} (lag = 1 year)')

Merged gdp_yearly_by_state.csv (lag = 1 year)


In [4]:
# monthly data: shift (today - last data month) months for each file
def get_monthly_shift(csv_path, m_df=None):
    if m_df is None:
        full = pd.read_csv(csv_path)
    else:
        full = m_df.copy()
    full['year'] = full['year'].astype(int)
    full['month'] = full['month'].astype(int)
    last = full.sort_values(['year', 'month']).iloc[-1]
    today = datetime.now()
    shift = (today.year * 12 + today.month) - (int(last['year']) * 12 + int(last['month']))
    return max(1, int(shift))

def merge_monthly(df, path, shift_months, by, m_df=None, date_col='approvaldate', state_col='borrstate'):
    if m_df is None:
        m_df = pd.read_csv(path)
    col = m_df.columns[-1]
    m_df['year'] = m_df['year'].astype(int)
    m_df['month'] = m_df['month'].astype(int)
    # if by state, groupby(state) ffill, otherwise directly ffill
    if by == 'state':
        m_df = m_df.sort_values(['state', 'year', 'month'])
        m_df[f'{col}_m_st'] = m_df.groupby('state')[col].ffill()
    else:
        m_df = m_df.sort_values(['year', 'month'])
        m_df[f'{col}_m'] = m_df[col].ffill()
    m_df = m_df.drop(columns=[col])
    df = df.copy()
    df['_lookup_date'] = pd.to_datetime(df[date_col]) - pd.DateOffset(months=shift_months)
    df['_lookup_y'] = df['_lookup_date'].dt.year
    df['_lookup_m'] = df['_lookup_date'].dt.month
    if by == 'national':
        df = df.merge(m_df, left_on=['_lookup_y', '_lookup_m'], right_on=['year', 'month'], how='left', validate='m:1')
        df = df.drop(columns=['_lookup_y', '_lookup_m', 'year', 'month'], errors='ignore')
    elif by == 'state':
        df = df.merge(m_df, left_on=['_lookup_y', '_lookup_m', state_col], right_on=['year', 'month', 'state'], how='left', validate='m:1')
        df = df.drop(columns=['_lookup_y', '_lookup_m', 'year', 'month', 'state'], errors='ignore')
    return df

# calculate lag and merge for each file
for fname in os.listdir(MACRO_DIR):
    if not fname.endswith('.csv') or 'gdp' in fname.lower():
        continue
    p = os.path.join(MACRO_DIR, fname)
    p_df = pd.read_csv(p)
    lag = get_monthly_shift(p)
    typ = 'state' if 'state' in fname.lower() else 'national'
    df_merge = merge_monthly(df_merge, None, lag, typ, m_df=p_df)

    # add growth rate
    value_col = p_df.columns[-1]
    growth_col = f"{value_col}_growth"
    if typ == 'state':
        p_df = p_df.sort_values(['state', 'year', 'month'])
        p_df[growth_col] = p_df.groupby('state')[value_col].pct_change(fill_method=None)
    else:
        p_df = p_df.sort_values(['year', 'month'])
        p_df[growth_col] = p_df[value_col].pct_change(fill_method=None)
    df_merge = merge_monthly(df_merge, None, lag, typ, m_df=p_df.drop(columns=[value_col]))

    print(f'Merged {fname} (lag = {lag} months)')

Merged employment_hours_earnings_by_state_monthly.csv (lag = 3 months)
Merged cpi_monthly.csv (lag = 2 months)
Merged hpi_monthly.csv (lag = 3 months)
Merged unemployment_rate_by_state_monthly.csv (lag = 3 months)
Merged crime_rate_by_state_monthly.csv (lag = 2 months)
Merged ppi_monthly.csv (lag = 2 months)


In [5]:
start_date = "1991-01-01"
tickers = {"sp500_ret": "^GSPC", "vix": "^VIX", "tnx": "^TNX", "t2y": "^IRX", "russell2000_ret": "^RUT"}

def yf_to_monthly(df_series):
    now = pd.Timestamp.now()
    cutoff = (now.replace(day=1) - pd.Timedelta(days=1))  # last day of last month
    s = df_series[df_series.index <= cutoff]
    return s.resample("ME").last()

today = datetime.now()
yf_shift = 1

for name, ticker in tickers.items():
    raw = yf.download(ticker, start=start_date, progress=False)
    s = raw['Close'].iloc[:, 0] if isinstance(raw.columns, pd.MultiIndex) else raw['Close']
    monthly = yf_to_monthly(s)
    if name in ['sp500_ret', 'russell2000_ret']:
        monthly = monthly.pct_change()
    last_month = monthly.index[-1]
    yf_shift = max(1, (today.year * 12 + today.month) - (last_month.year * 12 + last_month.month))
    m = monthly.reset_index()
    m.columns = ['date', name]
    m['year'] = m['date'].dt.year
    m['month'] = m['date'].dt.month
    m = m[['year', 'month', name]]
    df_merge = merge_monthly(df_merge, None, yf_shift, 'national', m_df=m)
    print(f'Merged {name} (lag = {yf_shift} months)')
df_merge['yield_spread_m'] = df_merge['tnx_m'] - df_merge['t2y_m']

Merged sp500_ret (lag = 1 months)
Merged vix (lag = 1 months)
Merged tnx (lag = 1 months)
Merged t2y (lag = 1 months)
Merged russell2000_ret (lag = 1 months)


In [6]:
df_merge = df_merge.drop(columns=['approval_year', 'approval_month', '_lookup_date'], errors='ignore')
df_merge.columns

Index(['asofdate', 'l2locid', 'borrname', 'borrcity', 'borrstate', 'cdc_state',
       'thirdpartylender_name', 'thirdpartylender_city',
       'thirdpartylender_state', 'thirdpartydollars', 'grossapproval',
       'approvaldate', 'firstdisbursementdate', 'processingmethod',
       'subprogram', 'terminmonths', 'naicscode', 'projectstate',
       'businesstype', 'businessage', 'loanstatus', 'paidinfulldate',
       'chargeoffdate', 'grosschargeoffamount', 'jobssupported',
       'collateralind', 'iffranchise', 'ifthirdparty', 'GDP_y_st',
       'GDP_growth_y_st', 'EHE_m_st', 'EHE_growth_m_st', 'CPI_m',
       'CPI_growth_m', 'HPI_m', 'HPI_growth_m', 'unemp_m_st',
       'unemp_growth_m_st', 'crime_rate_m_st', 'crime_rate_growth_m_st',
       'PPI_m', 'PPI_growth_m', 'sp500_ret_m', 'vix_m', 'tnx_m', 't2y_m',
       'russell2000_ret_m', 'yield_spread_m'],
      dtype='object')

In [7]:
df_merge.isna().sum().sort_values(ascending=False).head(20)

chargeoffdate             171821
paidinfulldate             72930
crime_rate_m_st             1488
crime_rate_growth_m_st      1488
l2locid                      124
GDP_y_st                      47
unemp_m_st                    47
unemp_growth_m_st             47
GDP_growth_y_st               47
EHE_m_st                      40
EHE_growth_m_st               40
asofdate                       0
borrstate                      0
cdc_state                      0
borrname                       0
borrcity                       0
terminmonths                   0
subprogram                     0
processingmethod               0
firstdisbursementdate          0
dtype: int64

In [8]:
df_merge.to_parquet('../data/data_merged.parquet')

## Time-Varying Survival Model

In [9]:
# ---- Time-varying panel merge (loan-month level) ----
def build_survival_end_event(df_input):
    """
    end_date: end of observation month (use chargeoffdate if available, otherwise use asofdate for right-censored)
    event: only example is charge-off event, can replace with your definition
    """
    x = df_input.copy()

    x['end_date'] = np.where(
            x['loanstatus'] == 'CHGOFF',
            x['chargeoffdate'],
            np.where(x['loanstatus'] == 'PIF', x['paidinfulldate'], x['asofdate'])
        )
    # event: 0=EXEMPT(censored), 1=CHGOFF(default), 2=PIF (competing risk)
    x['event'] = x['loanstatus'].map({'CHGOFF': 1, 'PIF': 2, 'EXEMPT': 0}).fillna(0).astype(int)

    return x


def expand_to_panel(df_input, id_col='loan_id'):
    """expand each loan into monthly intervals [start, stop) panel"""
    
    x = df_input.copy()

    if id_col not in x.columns:
        x[id_col] = np.arange(len(x)).astype(str)

    x['_start_m'] = x['firstdisbursementdate'].dt.to_period('M')
    x['_end_m'] = x['end_date'].dt.to_period('M')

    x['_n_months'] = (x['_end_m'] - x['_start_m']).apply(lambda p: p.n) + 1

    panel = x.loc[x.index.repeat(x['_n_months'])].copy().reset_index(drop=True)

    panel['month_start'] = panel.groupby(id_col).cumcount()
    panel['month_stop'] = panel['month_start'] + 1
    panel['loan_age'] = panel['month_start']

    panel['calendar_month'] = panel['_start_m'] + panel['month_start']

    panel['event_t'] = 0
    last_idx = panel.groupby(id_col).tail(1).index
    panel.loc[last_idx, 'event_t'] = panel.loc[last_idx, 'event']

    panel['panel_date'] = panel['calendar_month'].dt.to_timestamp(how='end')

    panel = panel.drop(columns=['_start_m', '_end_m', '_n_months'])
    panel = panel.reset_index(drop=True)

    return panel

In [10]:
# 1) construct panel from original loan data
base_surv = build_survival_end_event(df)
panel_df = expand_to_panel(base_surv, id_col='loan_id')

In [11]:
# 2) merge yearly state variable (GDP) with 1 year lag
gdp_path = os.path.join(MACRO_DIR, 'gdp_yearly_by_state.csv')
gdp = pd.read_csv(gdp_path)
gdp = gdp.sort_values(['state', 'year'])
gdp['GDP_y_st'] = gdp.groupby('state')['GDP'].ffill()
gdp['GDP_growth_y_st'] = gdp.groupby('state')['GDP_y_st'].pct_change()
gdp = gdp[['year', 'state', 'GDP_y_st', 'GDP_growth_y_st']]

panel_df['_lookup_year'] = panel_df['panel_date'].dt.year - 1
panel_df = panel_df.merge(
    gdp,
    left_on=['_lookup_year', 'borrstate'],
    right_on=['year', 'state'],
    how='left',
    validate='m:1'
).drop(columns=['_lookup_year', 'year', 'state'], errors='ignore')
print('Merged gdp_yearly_by_state.csv (lag = 1 year)')

# 3) monthly CSV variables: calculate lag automatically based on latest month available, and merge by state/national
for fname in os.listdir(MACRO_DIR):
    if not fname.endswith('.csv') or 'gdp' in fname.lower():
        continue

    p = os.path.join(MACRO_DIR, fname)
    p_df = pd.read_csv(p)
    lag = get_monthly_shift(None, p_df)
    typ = 'state' if 'state' in fname.lower() else 'national'

    # level values
    panel_df = merge_monthly(panel_df, None, lag, typ, m_df=p_df, date_col='panel_date', state_col='borrstate')

    # growth values
    value_col = p_df.columns[-1]
    growth_col = f'{value_col}_growth'
    p_growth = p_df.copy()
    if typ == 'state':
        p_growth = p_growth.sort_values(['state', 'year', 'month'])
        p_growth[growth_col] = p_growth.groupby('state')[value_col].pct_change(fill_method=None)
    else:
        p_growth = p_growth.sort_values(['year', 'month'])
        p_growth[growth_col] = p_growth[value_col].pct_change(fill_method=None)

    p_growth = p_growth.drop(columns=[value_col])
    panel_df = merge_monthly(panel_df, None, lag, typ, m_df=p_growth, date_col='panel_date', state_col='borrstate')

    print(f'Merged {fname} (lag = {lag} months)')

# 4) monthly yfinance variables, merge by monthly lag
start_date = '1991-01-01'
tickers = {
    'sp500_ret': '^GSPC',
    'vix': '^VIX',
    'tnx': '^TNX',
    't2y': '^IRX',
    'russell2000_ret': '^RUT'
}

for name, ticker in tickers.items():
    raw = yf.download(ticker, start=start_date, progress=False)
    s = raw['Close'].iloc[:, 0] if isinstance(raw.columns, pd.MultiIndex) else raw['Close']

    monthly = yf_to_monthly(s)
    if name in ['sp500_ret', 'russell2000_ret']:
        monthly = monthly.pct_change()

    last_m = monthly.index[-1]
    today = datetime.now()
    yf_lag = max(1, (today.year * 12 + today.month) - (last_m.year * 12 + last_m.month))

    tmp = monthly.reset_index()
    tmp.columns = ['date', name]
    tmp['year'] = tmp['date'].dt.year
    tmp['month'] = tmp['date'].dt.month
    tmp = tmp[['year', 'month', name]]

    panel_df = merge_monthly(panel_df, None, yf_lag, 'national', m_df=tmp, date_col='panel_date', state_col='borrstate')
    print(f'Merged {name} (lag = {yf_lag} months)')

panel_df['yield_spread_m'] = panel_df['tnx_m'] - panel_df['t2y_m']
panel_df = panel_df.drop(columns=['_lookup_date'], errors='ignore')

print(f'Panel number of rows: {len(panel_df):,}')
print(f'Number of loans: {panel_df["loan_id"].nunique():,}')
print(f'Average number of months per loan: {len(panel_df)/panel_df["loan_id"].nunique():.1f}')
panel_df.head()

Merged gdp_yearly_by_state.csv (lag = 1 year)
Merged employment_hours_earnings_by_state_monthly.csv (lag = 3 months)
Merged cpi_monthly.csv (lag = 2 months)
Merged hpi_monthly.csv (lag = 3 months)
Merged unemployment_rate_by_state_monthly.csv (lag = 3 months)
Merged crime_rate_by_state_monthly.csv (lag = 2 months)
Merged ppi_monthly.csv (lag = 2 months)
Merged sp500_ret (lag = 1 months)
Merged vix (lag = 1 months)
Merged tnx (lag = 1 months)
Merged t2y (lag = 1 months)
Merged russell2000_ret (lag = 1 months)
Panel number of rows: 17,653,820
Number of loans: 183,914
Average number of months per loan: 96.0


,asofdate,l2locid,borrname,borrcity,borrstate,cdc_state,thirdpartylender_name,thirdpartylender_city,thirdpartylender_state,thirdpartydollars,...,crime_rate_m_st,crime_rate_growth_m_st,PPI_m,PPI_growth_m,sp500_ret_m,vix_m,tnx_m,t2y_m,russell2000_ret_m,yield_spread_m
0,2025-12-31,188297.0,SAFEGUARD PROCUCTS INC,Other,PA,PA,Other,Other,NoThirdParty,0.0,...,36234.0,0.021338,121.2,0.004143,-0.026878,14.28,7.599,4.67,-0.004664,2.929
1,2025-12-31,188297.0,SAFEGUARD PROCUCTS INC,Other,PA,PA,Other,Other,NoThirdParty,0.0,...,32740.0,-0.096429,121.0,-0.001650,0.020834,14.56,7.793,5.02,-0.004295,2.773
2,2025-12-31,188297.0,SAFEGUARD PROCUCTS INC,Other,PA,PA,Other,Other,NoThirdParty,0.0,...,34038.0,0.039646,120.9,-0.000826,-0.039505,15.95,7.888,5.55,-0.042232,2.338
3,2025-12-31,188297.0,SAFEGUARD PROCUCTS INC,Other,PA,PA,Other,Other,NoThirdParty,0.0,...,30745.0,-0.096745,121.5,0.004963,0.012299,13.20,7.827,5.53,0.025015,2.297
4,2025-12-31,188297.0,SAFEGUARD PROCUCTS INC,Other,PA,PA,Other,Other,NoThirdParty,0.0,...,31969.0,0.039811,121.9,0.003292,0.024278,11.96,7.593,5.84,-0.014020,1.753


In [12]:
panel_df.to_parquet('../data/data_merged_tv.parquet')